# Pokemon Card Defect Detector v2 — YOLO Training

Train YOLOv11m to detect and localize defects on Pokemon cards using TAG Grading dataset.

**Dataset v2:** 23,116 images, 57,614 annotations, 7 classes

| Class | Count | % | Description |
|-------|-------|---|-------------|
| corner_wear | 23,694 | 41% | Whitening, bending, chipping at corners |
| surface_damage | 13,136 | 23% | Play wear, ink defects, print issues |
| edge_wear | 10,828 | 19% | Fraying, whitening along edges |
| crease | 3,698 | 6% | Folds, bends, wrinkles |
| scratch | 3,156 | 5% | Surface scratches, scuffing |
| dent | 2,827 | 5% | Dents and pits |
| stain | 275 | 0.5% | Stains, discoloration |

**Hardware:** RTX 4060 (8GB VRAM)

## 1. Setup

In [ ]:
# Install dependencies
!pip install -q ultralytics
!pip install -q pyyaml

In [ ]:
import os
import json
import random
import yaml
import numpy as np
from pathlib import Path
from collections import Counter
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from IPython.display import display, HTML

# Check GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

## 2. Upload Dataset

Upload your `tag_dataset.zip` (created by `scripts/scrape_tag.py --convert-only`).

**On your local machine, create the zip:**
```bash
cd data && zip -r tag_dataset.zip tag_dataset/
```

In [ ]:
# Local training: use the resized dataset (1280px, ~800MB)
# Copy tag_dataset_1280.zip to this machine and unzip, or use full dataset
DATASET_PATH = Path("data/tag_dataset_1280")  # resized for faster training

# Fallback to full dataset
if not DATASET_PATH.exists():
    DATASET_PATH = Path("data/tag_dataset")

# Unzip if needed
if not DATASET_PATH.exists():
    for zip_name in ["tag_dataset_1280.zip", "tag_dataset.zip"]:
        if Path(zip_name).exists():
            import zipfile
            with zipfile.ZipFile(zip_name, 'r') as z:
                z.extractall("data/")
            DATASET_PATH = Path("data") / zip_name.replace(".zip", "")
            break

DATASET_YAML = DATASET_PATH / "dataset.yaml"
assert DATASET_YAML.exists(), f"Dataset not found at {DATASET_YAML}"
print(f"Dataset: {DATASET_YAML}")

In [ ]:
# Fix dataset.yaml path to current directory
with open(DATASET_YAML) as f:
    cfg = yaml.safe_load(f)

cfg['path'] = str(DATASET_PATH.resolve())
if isinstance(cfg['names'], str):
    cfg['names'] = json.loads(cfg['names'])
# Ensure keys are ints
cfg['names'] = {int(k): v for k, v in cfg['names'].items()}

with open(DATASET_YAML, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

CLASS_NAMES = cfg['names']
NC = cfg['nc']
print(f"Classes ({NC}): {CLASS_NAMES}")

## 3. Dataset Exploration & Visualization

In [ ]:
# Count images and annotations per split
stats = {}
all_annotations = []

for split in ['train', 'val', 'test']:
    img_dir = DATASET_PATH / 'images' / split
    lbl_dir = DATASET_PATH / 'labels' / split
    images = sorted(img_dir.glob('*.jpg'))
    
    split_anns = 0
    split_empty = 0
    class_counts = Counter()
    
    for img_path in images:
        lbl_path = lbl_dir / (img_path.stem + '.txt')
        if not lbl_path.exists():
            continue
        content = lbl_path.read_text().strip()
        if not content:
            split_empty += 1
            continue
        for line in content.split('\n'):
            parts = line.strip().split()
            if len(parts) == 5:
                cls_id = int(parts[0])
                class_counts[cls_id] += 1
                split_anns += 1
                all_annotations.append({
                    'split': split,
                    'image': img_path.name,
                    'class': cls_id,
                    'x': float(parts[1]),
                    'y': float(parts[2]),
                    'w': float(parts[3]),
                    'h': float(parts[4]),
                })
    
    stats[split] = {
        'images': len(images),
        'annotations': split_anns,
        'empty': split_empty,
        'class_counts': dict(class_counts),
    }

for split, s in stats.items():
    print(f"{split:>5s}: {s['images']:5d} images, {s['annotations']:5d} annotations, {s['empty']:3d} negatives")

total_imgs = sum(s['images'] for s in stats.values())
total_anns = sum(s['annotations'] for s in stats.values())
print(f"{'TOTAL':>5s}: {total_imgs:5d} images, {total_anns:5d} annotations")

In [ ]:
# Class distribution bar chart
total_class_counts = Counter()
for s in stats.values():
    for cls_id, count in s['class_counts'].items():
        total_class_counts[cls_id] += count

classes = sorted(total_class_counts.keys())
counts = [total_class_counts[c] for c in classes]
names = [CLASS_NAMES[c] for c in classes]

colors = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#3498db', '#9b59b6', '#1abc9c']

fig, ax = plt.subplots(1, 1, figsize=(12, 5))
bars = ax.barh(names, counts, color=colors[:len(names)])
ax.set_xlabel('Number of Annotations')
ax.set_title('Defect Class Distribution')
for bar, count in zip(bars, counts):
    ax.text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
            f'{count}', va='center', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Defect location heatmap — where on the card do defects appear?
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for idx, cls_id in enumerate(sorted(CLASS_NAMES.keys())):
    row, col = idx // 4, idx % 4
    ax = axes[row][col]
    
    xs = [a['x'] for a in all_annotations if a['class'] == cls_id]
    ys = [a['y'] for a in all_annotations if a['class'] == cls_id]
    
    if xs:
        ax.hist2d(xs, ys, bins=20, range=[[0, 1], [0, 1]], cmap='YlOrRd')
    ax.set_title(f"{CLASS_NAMES[cls_id]} ({len(xs)})")
    ax.set_xlim(0, 1)
    ax.set_ylim(1, 0)  # flip Y to match image coords
    ax.set_aspect(600/825)  # card aspect ratio
    # Draw card outline
    rect = patches.Rectangle((0, 0), 1, 1, linewidth=2, edgecolor='black', facecolor='none')
    ax.add_patch(rect)

# Hide last subplot if odd number of classes
if len(CLASS_NAMES) < 8:
    axes[1][3].axis('off')

fig.suptitle('Defect Location Heatmaps (where on the card each defect type appears)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize sample cards with ground truth annotations
def draw_annotations(img_path, label_path, class_names, colors_map=None):
    """Draw bounding boxes on a card image."""
    img = Image.open(img_path)
    w_img, h_img = img.size
    draw = ImageDraw.Draw(img)
    
    if colors_map is None:
        colors_map = {
            0: '#e74c3c',  # corner_wear — red
            1: '#e67e22',  # edge_wear — orange
            2: '#f1c40f',  # surface_damage — yellow
            3: '#2ecc71',  # scratch — green
            4: '#3498db',  # crease — blue
            5: '#9b59b6',  # dent — purple
            6: '#1abc9c',  # stain — teal
        }
    
    content = label_path.read_text().strip()
    defects = []
    if content:
        for line in content.split('\n'):
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls_id = int(parts[0])
            xc, yc, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
            
            # Convert YOLO normalized to pixel coords
            x1 = int((xc - bw/2) * w_img)
            y1 = int((yc - bh/2) * h_img)
            x2 = int((xc + bw/2) * w_img)
            y2 = int((yc + bh/2) * h_img)
            
            color = colors_map.get(cls_id, '#ffffff')
            draw.rectangle([x1, y1, x2, y2], outline=color, width=4)
            
            label = class_names.get(cls_id, f'cls_{cls_id}')
            # Draw label background
            tw = len(label) * 20 + 10
            th = 30
            ly = max(0, y1 - th)
            draw.rectangle([x1, ly, x1 + tw, ly + th], fill=color)
            draw.text((x1 + 5, ly + 2), label, fill='white')
            
            defects.append(label)
    
    return img, defects


# Show 4x3 grid of annotated cards from training set
train_imgs = sorted((DATASET_PATH / 'images' / 'train').glob('*.jpg'))
# Pick images that have annotations (skip negatives)
annotated = []
for p in train_imgs:
    lbl = DATASET_PATH / 'labels' / 'train' / (p.stem + '.txt')
    if lbl.exists() and lbl.read_text().strip():
        annotated.append(p)

random.seed(42)
sample = random.sample(annotated, min(12, len(annotated)))

fig, axes = plt.subplots(3, 4, figsize=(24, 22))
for idx, img_path in enumerate(sample):
    row, col = idx // 4, idx % 4
    ax = axes[row][col]
    
    lbl_path = DATASET_PATH / 'labels' / 'train' / (img_path.stem + '.txt')
    annotated_img, defects = draw_annotations(img_path, lbl_path, CLASS_NAMES)
    
    # Resize for display
    annotated_img.thumbnail((800, 1100))
    ax.imshow(annotated_img)
    ax.set_title(f"{img_path.stem}\n{', '.join(defects)}", fontsize=9)
    ax.axis('off')

fig.suptitle('Training Set — Ground Truth Annotations', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Bbox size distribution per class
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Width distribution
for cls_id in sorted(CLASS_NAMES.keys()):
    widths = [a['w'] for a in all_annotations if a['class'] == cls_id]
    if widths:
        axes[0].hist(widths, bins=20, alpha=0.6, label=CLASS_NAMES[cls_id])
axes[0].set_title('Bbox Width Distribution')
axes[0].set_xlabel('Normalized Width')
axes[0].legend(fontsize=8)

# Height distribution
for cls_id in sorted(CLASS_NAMES.keys()):
    heights = [a['h'] for a in all_annotations if a['class'] == cls_id]
    if heights:
        axes[1].hist(heights, bins=20, alpha=0.6, label=CLASS_NAMES[cls_id])
axes[1].set_title('Bbox Height Distribution')
axes[1].set_xlabel('Normalized Height')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Annotations per image distribution
anns_per_img = Counter()
for a in all_annotations:
    anns_per_img[a['image']] += 1

counts_dist = Counter(anns_per_img.values())

fig, ax = plt.subplots(figsize=(10, 4))
x = sorted(counts_dist.keys())
y = [counts_dist[k] for k in x]
ax.bar(x, y, color='#3498db')
ax.set_xlabel('Defects per Image')
ax.set_ylabel('Number of Images')
ax.set_title(f'Defects per Image Distribution (avg: {np.mean(list(anns_per_img.values())):.1f})')
plt.tight_layout()
plt.show()

## 4. Train YOLOv11m

In [ ]:
from ultralytics import YOLO

# Load YOLOv11m pretrained on COCO
model = YOLO('yolo11m.pt')
print(f"Model: {model.model_name}")
print(f"Parameters: {sum(p.numel() for p in model.model.parameters()) / 1e6:.1f}M")

In [ ]:
# RTX 4060 (8GB VRAM): imgsz=640, batch=16
# RTX 4070 (12GB VRAM): imgsz=640, batch=24 or imgsz=1280, batch=4
IMGSZ = 640
BATCH = 16

results = model.train(
    data=str(DATASET_YAML),
    epochs=100,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=20,
    lr0=0.01,
    lrf=0.01,
    optimizer='AdamW',
    weight_decay=0.0005,
    # Card-specific augmentation
    mosaic=1.0,
    flipud=0.0,         # cards have orientation — don't flip vertically
    fliplr=0.5,
    degrees=3.0,         # slight rotation only
    scale=0.3,
    hsv_h=0.015,
    hsv_s=0.4,
    hsv_v=0.4,
    translate=0.1,
    perspective=0.0,     # cards are rectified — no perspective warp
    shear=0.0,
    erasing=0.1,         # random erasing augmentation
    close_mosaic=10,
    warmup_epochs=3,
    cos_lr=True,
    # Class imbalance mitigation
    cls=1.5,             # increase classification loss weight (stain class)
    # Output
    project='runs/defect',
    name='train_v2',
    exist_ok=True,
    save=True,
    plots=True,
    verbose=True,
)

## 5. Training Results

In [ ]:
# Load training results
import pandas as pd

TRAIN_DIR = Path('runs/defect/train_v2')
results_csv = TRAIN_DIR / 'results.csv'

if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    # Loss curves
    for i, col in enumerate(['train/box_loss', 'train/cls_loss', 'train/dfl_loss']):
        if col in df.columns:
            axes[0][i].plot(df['epoch'], df[col], 'b-', alpha=0.7, label='Train')
            val_col = col.replace('train/', 'val/')
            if val_col in df.columns:
                axes[0][i].plot(df['epoch'], df[val_col], 'r-', alpha=0.7, label='Val')
            axes[0][i].set_title(col.split('/')[-1].replace('_', ' ').title())
            axes[0][i].set_xlabel('Epoch')
            axes[0][i].legend()
            axes[0][i].grid(True, alpha=0.3)
    
    # mAP curves
    metrics = ['metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'metrics/precision(B)']
    titles = ['mAP@50', 'mAP@50-95', 'Precision']
    for i, (col, title) in enumerate(zip(metrics, titles)):
        if col in df.columns:
            axes[1][i].plot(df['epoch'], df[col], 'g-', linewidth=2)
            axes[1][i].set_title(title)
            axes[1][i].set_xlabel('Epoch')
            axes[1][i].set_ylim(0, 1)
            axes[1][i].grid(True, alpha=0.3)
            best_idx = df[col].idxmax()
            best_val = df[col].iloc[best_idx]
            axes[1][i].axhline(y=best_val, color='r', linestyle='--', alpha=0.5)
            axes[1][i].text(0, best_val + 0.02, f'Best: {best_val:.3f}', color='r', fontweight='bold')
    
    fig.suptitle('Training Progress — v2 (57K annotations)', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Print final metrics
    print("\n" + "="*50)
    print("FINAL METRICS")
    print("="*50)
    last = df.iloc[-1]
    for col in ['metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'metrics/precision(B)', 'metrics/recall(B)']:
        if col in df.columns:
            name = col.split('/')[-1].replace('(B)', '')
            print(f"  {name:20s} {last[col]:.4f}")
else:
    print('No results.csv found — training not complete yet')

In [ ]:
# Show YOLO's built-in training plots
for plot_name in ['confusion_matrix_normalized.png', 'results.png', 'PR_curve.png', 'F1_curve.png']:
    plot_path = TRAIN_DIR / plot_name
    if plot_path.exists():
        img = Image.open(plot_path)
        fig, ax = plt.subplots(figsize=(12, 8))
        ax.imshow(img)
        ax.set_title(plot_name.replace('.png', '').replace('_', ' ').title(), fontsize=14)
        ax.axis('off')
        plt.tight_layout()
        plt.show()

## 6. Validation — Predictions on Test Set

In [ ]:
# Load best model
best_model_path = TRAIN_DIR / 'weights' / 'best.pt'
if not best_model_path.exists():
    print(f"Best model not found at {best_model_path}")
else:
    best_model = YOLO(str(best_model_path))
    print(f"Loaded: {best_model_path}")
    
    # Run validation on test set
    val_results = best_model.val(
        data=str(DATASET_YAML),
        split='test',
        verbose=True,
    )
    
    print(f"\nTest mAP50: {val_results.box.map50:.4f}")
    print(f"Test mAP50-95: {val_results.box.map:.4f}")
    
    # Per-class AP
    print("\nPer-class AP@50:")
    for i, ap in enumerate(val_results.box.ap50):
        name = CLASS_NAMES.get(i, f'class_{i}')
        print(f"  {name:20s} {ap:.4f}")

In [ ]:
# Side-by-side: Ground Truth vs Predictions on test images
test_imgs = sorted((DATASET_PATH / 'images' / 'test').glob('*.jpg'))

# Pick images with annotations
test_annotated = []
for p in test_imgs:
    lbl = DATASET_PATH / 'labels' / 'test' / (p.stem + '.txt')
    if lbl.exists() and lbl.read_text().strip():
        test_annotated.append(p)

random.seed(123)
sample_test = random.sample(test_annotated, min(8, len(test_annotated)))

fig, axes = plt.subplots(len(sample_test), 2, figsize=(16, 8 * len(sample_test)))
if len(sample_test) == 1:
    axes = [axes]

pred_colors = {
    0: (231, 76, 60),    # corner_wear — red
    1: (230, 126, 34),   # edge_wear — orange
    2: (241, 196, 15),   # surface_damage — yellow
    3: (46, 204, 113),   # scratch — green
    4: (52, 152, 219),   # crease — blue
    5: (155, 89, 182),   # dent — purple
    6: (26, 188, 156),   # stain — teal
}

for idx, img_path in enumerate(sample_test):
    # Ground truth
    lbl_path = DATASET_PATH / 'labels' / 'test' / (img_path.stem + '.txt')
    gt_img, gt_defects = draw_annotations(img_path, lbl_path, CLASS_NAMES)
    gt_img.thumbnail((800, 1100))
    axes[idx][0].imshow(gt_img)
    axes[idx][0].set_title(f'Ground Truth\n{img_path.stem}\n{\', \'.join(gt_defects)}', fontsize=10)
    axes[idx][0].axis('off')
    
    # Predictions
    pred_results = best_model.predict(str(img_path), conf=0.25, verbose=False)
    pred_img = Image.open(img_path).copy()
    draw = ImageDraw.Draw(pred_img)
    pred_defects = []
    
    if pred_results and len(pred_results[0].boxes):
        for box in pred_results[0].boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            cls_id = int(box.cls[0].cpu())
            conf = float(box.conf[0].cpu())
            color = pred_colors.get(cls_id, (255, 255, 255))
            
            draw.rectangle([x1, y1, x2, y2], outline=color, width=4)
            label = f"{CLASS_NAMES.get(cls_id, '?')} {conf:.2f}"
            tw = len(label) * 18 + 10
            ly = max(0, y1 - 28)
            draw.rectangle([x1, ly, x1 + tw, ly + 28], fill=color)
            draw.text((x1 + 5, ly + 2), label, fill='white')
            pred_defects.append(f"{CLASS_NAMES.get(cls_id, '?')} ({conf:.2f})")
    
    pred_img.thumbnail((800, 1100))
    axes[idx][1].imshow(pred_img)
    pred_str = ', '.join(pred_defects) if pred_defects else 'No defects found'
    axes[idx][1].set_title(f'Predictions\n{pred_str}', fontsize=10)
    axes[idx][1].axis('off')

fig.suptitle('Ground Truth vs Model Predictions (Test Set)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Per-class performance bar chart
if 'val_results' in dir():
    ap50s = val_results.box.ap50
    
    fig, ax = plt.subplots(figsize=(12, 5))
    x = range(len(ap50s))
    class_labels = [CLASS_NAMES.get(i, f'cls_{i}') for i in range(len(ap50s))]
    bar_colors = [colors[i % len(colors)] for i in range(len(ap50s))]
    
    bars = ax.bar(x, ap50s, color=bar_colors)
    ax.set_xticks(x)
    ax.set_xticklabels(class_labels, rotation=30, ha='right')
    ax.set_ylabel('AP@50')
    ax.set_title(f'Per-Class AP@50 on Test Set (Overall mAP50: {val_results.box.map50:.3f})')
    ax.set_ylim(0, 1)
    ax.axhline(y=val_results.box.map50, color='red', linestyle='--', label=f'Mean: {val_results.box.map50:.3f}')
    ax.legend()
    
    for bar, ap in zip(bars, ap50s):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{ap:.3f}', ha='center', fontweight='bold', fontsize=10)
    
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

## 7. Inference Demo — Grade a Card

In [ ]:
def grade_card_defects(model, image_path, conf_threshold=0.25):
    """Run defect detection and generate a visual report."""
    img = Image.open(image_path)
    w_img, h_img = img.size
    
    results = model.predict(str(image_path), conf=conf_threshold, verbose=False)
    
    defects = []
    if results and len(results[0].boxes):
        for box in results[0].boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            cls_id = int(box.cls[0].cpu())
            conf = float(box.conf[0].cpu())
            
            # Map confidence to severity
            if conf > 0.7:
                severity = 'HIGH'
                severity_color = '#e74c3c'
            elif conf > 0.4:
                severity = 'MEDIUM'
                severity_color = '#f39c12'
            else:
                severity = 'LOW'
                severity_color = '#2ecc71'
            
            # Determine location on card
            cx = (x1 + x2) / 2 / w_img
            cy = (y1 + y2) / 2 / h_img
            if cy < 0.15:
                loc_v = 'top'
            elif cy > 0.85:
                loc_v = 'bottom'
            else:
                loc_v = 'center'
            if cx < 0.2:
                loc_h = 'left'
            elif cx > 0.8:
                loc_h = 'right'
            else:
                loc_h = 'center'
            location = f"{loc_v}-{loc_h}".replace('center-center', 'center')
            
            defects.append({
                'type': CLASS_NAMES.get(cls_id, f'class_{cls_id}'),
                'confidence': conf,
                'severity': severity,
                'severity_color': severity_color,
                'location': location,
                'bbox': [x1, y1, x2, y2],
                'cls_id': cls_id,
            })
    
    # Sort by confidence descending
    defects.sort(key=lambda d: d['confidence'], reverse=True)
    
    # Create visualization
    fig = plt.figure(figsize=(18, 12))
    
    # Left: annotated card
    ax_img = fig.add_subplot(1, 2, 1)
    draw_img = img.copy()
    draw = ImageDraw.Draw(draw_img)
    
    for d in defects:
        x1, y1, x2, y2 = d['bbox']
        color = pred_colors.get(d['cls_id'], (255, 255, 255))
        draw.rectangle([x1, y1, x2, y2], outline=color, width=5)
        label = f"{d['type']} ({d['confidence']:.0%})"
        tw = len(label) * 22
        ly = max(0, y1 - 35)
        draw.rectangle([x1, ly, x1 + tw, ly + 35], fill=color)
        draw.text((x1 + 5, ly + 3), label, fill='white')
    
    draw_img.thumbnail((900, 1200))
    ax_img.imshow(draw_img)
    ax_img.set_title('Detected Defects', fontsize=14, fontweight='bold')
    ax_img.axis('off')
    
    # Right: defect report
    ax_report = fig.add_subplot(1, 2, 2)
    ax_report.axis('off')
    
    report_text = f"DEFECT DETECTION REPORT\n{'='*35}\n\n"
    report_text += f"Total defects found: {len(defects)}\n\n"
    
    if not defects:
        report_text += "No defects detected!\n"
        report_text += "This card appears to be in excellent condition.\n"
    else:
        for i, d in enumerate(defects, 1):
            report_text += f"Defect #{i}:\n"
            report_text += f"  Type:       {d['type']}\n"
            report_text += f"  Confidence: {d['confidence']:.1%}\n"
            report_text += f"  Severity:   {d['severity']}\n"
            report_text += f"  Location:   {d['location']}\n\n"
        
        # Summary by pillar
        corner_defects = [d for d in defects if d['type'] == 'corner_wear']
        edge_defects = [d for d in defects if d['type'] == 'edge_wear']
        surface_defects = [d for d in defects if d['type'] not in ('corner_wear', 'edge_wear')]
        
        report_text += f"{'='*35}\nSUMMARY BY PILLAR\n{'='*35}\n"
        report_text += f"  Corners:  {len(corner_defects)} defect(s)\n"
        report_text += f"  Edges:    {len(edge_defects)} defect(s)\n"
        report_text += f"  Surface:  {len(surface_defects)} defect(s)\n"
    
    ax_report.text(0.05, 0.95, report_text, transform=ax_report.transAxes,
                   fontsize=12, verticalalignment='top', fontfamily='monospace',
                   bbox=dict(boxstyle='round', facecolor='#f8f9fa', edgecolor='#dee2e6'))
    
    plt.tight_layout()
    plt.show()
    
    return defects

In [ ]:
# Demo: grade 3 test cards
if 'best_model' in dir():
    demo_cards = random.sample(test_annotated, min(3, len(test_annotated)))
    for card_path in demo_cards:
        print(f"\n{'='*60}")
        print(f"Card: {card_path.stem}")
        print(f"{'='*60}")
        defects = grade_card_defects(best_model, card_path)

## 8. Export to ONNX

In [ ]:
if 'best_model' in dir():
    # Export to ONNX (match training imgsz)
    best_model.export(format='onnx', imgsz=IMGSZ, simplify=True)
    
    onnx_path = TRAIN_DIR / 'weights' / 'best.onnx'
    if onnx_path.exists():
        size_mb = onnx_path.stat().st_size / 1024 / 1024
        print(f"\nONNX model: {onnx_path}")
        print(f"Size: {size_mb:.1f} MB")
        print(f"Training imgsz: {IMGSZ}")
        print(f"\nCopy this file to your server:")
        print(f"  models/yolo_defect/defect_detector.onnx")

In [ ]:
# Download model (Colab)
# from google.colab import files
# files.download(str(onnx_path))
# files.download(str(TRAIN_DIR / 'weights' / 'best.pt'))

## 9. Summary

### What we trained
- **Model:** YOLOv11m fine-tuned for card defect detection
- **Dataset:** TAG Grading data — professional card grading annotations
- **Classes:** 7 defect types (corner_wear, edge_wear, surface_damage, scratch, crease, dent, stain)

### Next steps
1. Download `best.onnx` → place in `models/yolo_defect/`
2. Create `src/defect_detector.py` for ONNX inference
3. Integrate into `/gemini/grade` API endpoint
4. Active learning: use model predictions to find/fix bad annotations → retrain v2